# Analyse Exploratoire des Données (EDA)
Ce notebook a pour but d'explorer le jeu de données `dataset_ProjetML_2026.csv` brut. Nous allons analyser la distribution de nos variables, détecter les outliers, évaluer les corrélations et diagnostiquer la nature des valeurs manquantes, sans appliquer aucune transformation ou imputation pour le moment.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings("ignore")

NUMERIC = ['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite', 'Prix_Revente']
TARGET = 'Categorie'
SOURCE_COL = 'Source' 

## Chargement des données
Chargement du fichier brut depuis notre répertoire `data/raw` et vérification de la structure globale du DataFrame.


In [2]:
df = pd.read_csv("../data/raw/dataset_ProjetML_2026.csv")
print(f"Shape : {df.shape}\n")
print("Types des colonnes :")
print(df.dtypes)
print("\nAperçu (head) :")
display(df.head())
print("\nDescription statistique des variables numériques :")
display(df[NUMERIC].describe().round(3))

Shape : (10500, 9)

Types des colonnes :
Poids               float64
Volume              float64
Conductivite        float64
Opacite             float64
Rigidite            float64
Prix_Revente        float64
Categorie            object
Source               object
Rapport_Collecte     object
dtype: object

Aperçu (head) :


,Poids,Volume,Conductivite,Opacite,Rigidite,Prix_Revente,Categorie,Source,Rapport_Collecte
0,16.708780,70.940977,0.0,1.0,1.0,0.835439,Papier,NaN,Lot de papier récupéré dans un site non rensei...
1,47.277476,64.702925,0.0,NaN,3.0,4.727748,Plastique,Usine_A,"Lot plastique à l'Usine A. Volume 64.7 L, poid..."
2,NaN,317.415183,0.0,NaN,9.0,4.211790,Verre,Usine_B,Bris de verre ou contenants en provenance de l...
3,NaN,21.474391,0.0,NaN,1.0,0.442067,Papier,Centre_Tri,Feuilles et cartons collectés au Centre de Tri...
4,NaN,59.462176,0.0,1.0,NaN,0.723004,Papier,Usine_B,Déchet de type papier identifié à l'Usine B. V...



Description statistique des variables numériques :


,Poids,Volume,Conductivite,Opacite,Rigidite,Prix_Revente
count,9471.000,9960.000,9483.000,9465.000,9942.000,9964.000
mean,77.797,144.408,0.208,1.160,5.887,58.588
std,127.847,136.384,0.379,5.493,3.087,720.059
min,-99.000,-26.808,0.000,0.000,1.000,-50.000
25%,19.752,44.437,0.000,0.196,3.000,1.394
50%,39.193,88.084,0.000,0.553,5.000,4.135
75%,130.498,240.200,0.000,1.000,9.000,6.782
max,2334.219,554.107,0.999,55.000,10.000,9999.000


## Distribution de la variable cible Categorie
Nous visualisons ici la répartition de notre variable cible pour comprendre s'il y a un déséquilibre de classes. Note : la catégorie cible peut contenir des valeurs nulles (non labellisées).


In [3]:
cat_counts = df[TARGET].value_counts(dropna=False).reset_index()
cat_counts.columns = [TARGET, 'Compte']
# Remplacer les NaN par une chaîne de caractères pour affichage
cat_counts[TARGET] = cat_counts[TARGET].fillna("Non Labellisé")

fig_target = px.bar(
    cat_counts, 
    x=TARGET, y='Compte', 
    title="Distribution de la variable cible (Categorie)",
    text='Compte',
    color=TARGET,
    height=400
)
fig_target.update_traces(textposition='outside')
fig_target.show()

## Analyse des valeurs manquantes
Il est crucial d'identifier quelles colonnes contiennent des valeurs manquantes et dans quelles proportions pour cibler nos futures stratégies d'imputation.


In [4]:
missing_stats = df.isnull().sum().reset_index()
missing_stats.columns = ['Colonne', 'Valeurs Manquantes']
missing_stats['Pourcentage (%)'] = (missing_stats['Valeurs Manquantes'] / len(df)) * 100
missing_stats = missing_stats[missing_stats['Valeurs Manquantes'] > 0].sort_values(by='Pourcentage (%)', ascending=False)
display(missing_stats.round(2))

,Colonne,Valeurs Manquantes,Pourcentage (%)
3,Opacite,1035,9.86
0,Poids,1029,9.80
2,Conductivite,1017,9.69
4,Rigidite,558,5.31
1,Volume,540,5.14
5,Prix_Revente,536,5.10
7,Source,536,5.10
6,Categorie,514,4.90


In [5]:
missing_matrix = df[NUMERIC].isnull().astype(int)
fig_missing = px.imshow(
    missing_matrix.T,
    color_continuous_scale=["#2ecc71", "#e74c3c"],
    title="Carte des valeurs manquantes des variables numériques (Rouge = Manquant)",
    labels={"x": "Observations", "y": "Variable", "color": "Manquant"},
    aspect="auto",
    height=400
)
fig_missing.update_layout(coloraxis_showscale=False)
fig_missing.show()

## Diagnostic MCAR / MAR / MNAR
Dans cette étape d'exploration, nous effectuons un premier diagnostic visuel et logique des mécanismes de données manquantes :

- **MCAR (Missing Completely At Random)** : Probabilité de manquer indépendante des données (observées ou non).
- **MAR (Missing At Random)** : Probabilité de manquer liée à d'autres variables observées (ex: on peut constater que 'Poids' est souvent manquant pour une certaine 'Source').
- **MNAR (Missing Not At Random)** : Probabilité de manquer liée à la valeur manquante elle-même. Par exemple, pour `Prix_Revente`, la présence de codes sentinelles comme 9999 ou -50 indique que la donnée manquante a une signification particulière, ce qui relève du MNAR.


In [6]:
from scipy import stats

def littles_mcar_test(data):
    df_num = data.copy()
    n, p = df_num.shape
    R = df_num.isnull().astype(int)
    patterns = R.drop_duplicates()
    grand_mean = df_num.mean()
    grand_cov  = df_num.cov()
    chi2_stat, df_freedom = 0.0, 0
    for _, pattern in patterns.iterrows():
        observed_cols = pattern[pattern == 0].index.tolist()
        if not observed_cols: continue
        mask = (R == pattern.values).all(axis=1)
        subset = df_num.loc[mask, observed_cols]
        if len(subset) < 2: continue
        diff = (subset.mean() - grand_mean[observed_cols]).values
        try:
            cov_inv = np.linalg.pinv(grand_cov.loc[observed_cols, observed_cols].values)
            chi2_stat += len(subset) * diff @ cov_inv @ diff
            df_freedom += len(observed_cols)
        except: continue
    p_value = 1 - stats.chi2.cdf(chi2_stat, df=max(df_freedom - p, 1))
    return {"chi2": round(chi2_stat, 4), "df": df_freedom, "p_value": round(p_value, 4)}

NUMERIC = ['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite', 'Prix_Revente']
res = littles_mcar_test(df[NUMERIC])
print(f"Little's MCAR Test → chi²={res['chi2']}, p={res['p_value']}")
print("→ H0 non rejetée (MCAR)" if res['p_value'] > 0.05 else "→ H0 rejetée : MAR ou MNAR probable")


Little's MCAR Test → chi²=184.2081, p=0.0052
→ H0 rejetée : MAR ou MNAR probable


In [7]:
print("Taux manquants Poids par Source :")
display(df.groupby('Source')['Poids'].apply(lambda x: x.isnull().mean()*100).round(2).rename("% NaN"))

print("\nTaux manquants Poids par Categorie :")
display(df.groupby('Categorie')['Poids'].apply(lambda x: x.isnull().mean()*100).round(2).rename("% NaN"))

print(f"\nMNAR → Prix_Revente=9999 : {(df['Prix_Revente']==9999).sum()} cas")
print(f"MNAR → Prix_Revente=-50  : {(df['Prix_Revente']==-50).sum()} cas")


Taux manquants Poids par Source :


Source
Centre_Tri             9.31
Collecte_Citoyenne    10.10
Usine_A               10.11
Usine_B                9.90
Name: % NaN, dtype: float64


Taux manquants Poids par Categorie :


Categorie
Métal        11.15
Papier        8.71
Plastique    10.59
Verre         8.66
Name: % NaN, dtype: float64


MNAR → Prix_Revente=9999 : 52 cas
MNAR → Prix_Revente=-50  : 10 cas


## Distribution des variables numériques
L'observation de la forme des distributions (symétrie, asymétrie, pics) est nécessaire pour déterminer si certaines variables nécessiteront des transformations log ou spécifiques ultérieurement.


In [8]:
fig_hist = make_subplots(rows=2, cols=3, subplot_titles=NUMERIC)
for i, col in enumerate(NUMERIC):
    r, c = divmod(i, 3)
    fig_hist.add_trace(
        go.Histogram(x=df[col].dropna(), name=col, marker_color="#3498db", nbinsx=40, showlegend=False),
        row=r+1, col=c+1
    )
fig_hist.update_layout(title_text="Histogrammes des variables numériques", height=600)
fig_hist.show()

## Détection des Outliers
L'utilisation de diagrammes en boîte (boxplots) nous permet d'identifier visuellement la présence de valeurs atypiques (outliers) ou de codes d'erreur (comme un opacité de 55.0 hors norme) avant tout nettoyage.


In [9]:
fig_box = make_subplots(rows=2, cols=3, subplot_titles=NUMERIC)
for i, col in enumerate(NUMERIC):
    r, c = divmod(i, 3)
    fig_box.add_trace(
        go.Box(y=df[col].dropna(), name=col, marker_color="#e74c3c", showlegend=False),
        row=r+1, col=c+1
    )
fig_box.update_layout(title_text="Boxplots des variables numériques (Détection d'outliers)", height=600)
fig_box.show()

## Matrice de corrélation
Nous cherchons ici à évaluer la multicolinéarité potentielle entre nos variables explicatives continues. Des corrélations très fortes pourraient nécessiter la suppression de certaines variables.


In [10]:
corr_matrix = df[NUMERIC].corr()
fig_corr = px.imshow(
    corr_matrix, 
    text_auto=True, 
    color_continuous_scale='RdBu_r', 
    zmin=-1, zmax=1,
    title="Matrice de corrélation de Pearson",
    height=500
)
fig_corr.show()

## Analyse de la colonne Source
Nous observons la répartition de la provenance (la source) des différentes matières collectées.


In [11]:
source_counts = df[SOURCE_COL].value_counts(dropna=False).reset_index()
source_counts.columns = [SOURCE_COL, 'Compte']
source_counts[SOURCE_COL] = source_counts[SOURCE_COL].fillna("Non Renseigné")

fig_source = px.bar(
    source_counts,
    x=SOURCE_COL, y='Compte',
    title="Répartition par Source",
    text='Compte',
    color=SOURCE_COL,
    height=400
)
fig_source.update_traces(textposition='outside')
fig_source.show()

## Distribution de Prix_Revente
Un focus particulier est mis sur la variable `Prix_Revente` afin de mettre en évidence sa dissymétrie (skewness) due notamment à l'impact potentiel des codes sentinelles non encore traités.


In [12]:
skewness_val = df['Prix_Revente'].skew()
fig_prix = px.histogram(
    df, x='Prix_Revente', 
    title=f"Distribution de Prix_Revente (Skewness: {skewness_val:.2f})",
    nbins=50, 
    color_discrete_sequence=['#9b59b6'],
    height=400
)
fig_prix.show()

## Conclusion EDA
Lors de cette phase d'exploration, nous avons pu constater :
- La présence de codes sentinelles flagrants (ex: `Prix_Revente` = 9999, `Opacite` > 1).
- Des asymétries et des valeurs manquantes variables selon les colonnes qu'il faudra combler par imputation.
- Que certaines caractéristiques, bien qu'ayant des comportements atypiques (ex: conductivité à 0), reflètent fidèlement la réalité de matériaux isolants.

Le script de *preprocessing* se chargera de traiter l'intégralité de ces anomalies (imputations, standardisation, découpage stratifié).
